# MCP Concepts 04: Connecting to a Real Public MCP Server (Streamable HTTP)

## Problem card

- **What's different here:** every server in notebooks 01-03 was ours --
  code we wrote, ran, and fully controlled, over stdio. This notebook
  connects to **DeepWiki's public MCP server**
  (`https://mcp.deepwiki.com/mcp`) -- a real, free, no-authentication-required
  server run by a third party, reachable only over the network, that we have
  no control over and didn't write a single line of.
- **Why this matters:** stdio is fine for local tools you own. The moment
  you connect to *anyone else's* server, you inherit a genuinely different
  set of risks and constraints -- covered in this notebook's second half.
- **What the server does:** DeepWiki generates and hosts documentation for
  public GitHub repositories; its MCP server exposes that as `ask_question`
  (an AI-synthesized answer grounded in a repo's wiki), `read_wiki_contents`,
  and `read_wiki_structure`.
- **Success criteria:** a real network round trip to a server we don't run,
  correctly wired into a LangGraph agent, with a documented failure we hit
  and fixed along the way -- not a hypothetical.

## stdio vs. Streamable HTTP, side by side

| | stdio (notebooks 01-03) | Streamable HTTP (this notebook) |
|---|---|---|
| Who starts the server | The client, as a subprocess | Already running, owned by someone else |
| Trust boundary | None -- it's your own code | Real -- you're trusting a third party's server and its outputs |
| Network exposure | None | A real HTTPS request, same as any API call |
| Client setup | `StdioServerParameters(command=..., args=...)` | `{"url": "...", "transport": "streamable_http"}` |

```mermaid
sequenceDiagram
    participant C as Client (this notebook)
    participant S as DeepWiki MCP Server (mcp.deepwiki.com, not ours)

    C->>S: HTTPS POST /mcp -- initialize
    S-->>C: capabilities, protocol version
    C->>S: list_tools
    S-->>C: ask_question, read_wiki_contents, read_wiki_structure
    C->>S: call_tool("ask_question", {repoName, question})
    S-->>C: synthesized answer (real network round trip)
```

In [1]:
import os
import sys
import warnings
from typing import TypedDict, Annotated
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore")
load_dotenv(find_dotenv(usecwd=True))

sys.path.insert(0, os.getcwd())
import shared

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI

DEEPWIKI_URL = "https://mcp.deepwiki.com/mcp"
print(f"Public server: {DEEPWIKI_URL}")

Public server: https://mcp.deepwiki.com/mcp


## Step 1: raw protocol inspection, same as notebook 01, just over HTTP to a
server we don't run

In [2]:
async def inspect_public_server():
    async with streamablehttp_client(DEEPWIKI_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            print(f"Connected to: {init.serverInfo.name} (protocol {init.protocolVersion})")
            tools = await session.list_tools()
            for t in tools.tools:
                print(f"- {t.name}: {t.description[:100]}")

await inspect_public_server()

Connected to: DeepWiki (protocol 2025-11-25)
- ask_question: Ask any question about a GitHub repository and get an AI-powered, context-grounded response.
- read_wiki_contents: View documentation about a GitHub repository.
- read_wiki_structure: Get a list of documentation topics for a GitHub repository.


## Step 2: a real failure -- context overflow from raw wiki content

The first, naive attempt below binds **all three** DeepWiki tools to the
agent and asks a broad question. What actually happened the first time this
was run: the model called `read_wiki_contents`, which returned the *entire*
wiki for the repository -- **over 167,000 tokens** -- blowing past
gpt-4o-mini's 128,000-token context window and raising a real
`OpenAIContextOverflowError`. This is left in as a genuine lesson, not
smoothed over: a public MCP server's tool can return an amount of data your
model simply cannot consume in one call, and nothing about the MCP protocol
itself prevents that.

In [3]:
mcp_client_all = MultiServerMCPClient({
    "deepwiki": {"url": DEEPWIKI_URL, "transport": "streamable_http"},
})
all_deepwiki_tools = await mcp_client_all.get_tools()
for t in all_deepwiki_tools:
    print(f"- {t.name}")

- ask_question
- read_wiki_contents
- read_wiki_structure


## Step 3: the fix -- scope down to the tool that fits the job

`ask_question` returns an AI-synthesized, already-concise answer instead of
raw wiki dump -- exactly the tool shape you'd want for a chat-style agent.
The fix here isn't a framework setting; it's the same lesson as notebook
03's fix: **be deliberate about which tools from a server you actually bind**,
especially a server you don't control and whose tools can return
wildly different amounts of data.

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

scoped_tools = [t for t in all_deepwiki_tools if t.name == "ask_question"]
print(f"Bound tools: {[t.name for t in scoped_tools]}")

llm_with_tools = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), temperature=0
).bind_tools(scoped_tools)


async def agent_node(state: AgentState) -> dict:
    response = await llm_with_tools.ainvoke(state["messages"])
    return {"messages": [response]}


builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(scoped_tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": END})
builder.add_edge("tools", "agent")
public_mcp_agent = builder.compile()
print("Graph compiled.")

Bound tools: ['ask_question']
Graph compiled.


In [5]:
with shared.Timer() as t:
    result = await public_mcp_agent.ainvoke({
        "messages": [{"role": "user", "content":
            "Using the langchain-ai/langgraph GitHub repo, briefly: what is LangGraph used for?"}]
    })

for m in result["messages"]:
    tc = getattr(m, "tool_calls", None)
    if tc:
        for c in tc:
            print(f"tool_call -> {c['name']}({ {k: v for k, v in c['args'].items()} })")
    elif getattr(m, "content", None):
        print(f"\n[{type(m).__name__}] {m.content[:600]}")

print(f"\nwall clock: {t.elapsed_s:.2f}s (includes a real network round trip to a public server)")


[HumanMessage] Using the langchain-ai/langgraph GitHub repo, briefly: what is LangGraph used for?
tool_call -> ask_question({'repoName': 'langchain-ai/langgraph', 'question': 'What is LangGraph used for?'})

[ToolMessage] [{'type': 'text', 'text': 'LangGraph is a low-level orchestration framework designed for building stateful, multi-actor applications with Large Language Models (LLMs). It provides the infrastructure for durable execution, streaming, human-in-the-loop interactions, persistence, and memory in long-running agentic workflows. \n\n## Core Purpose and Capabilities\nLangGraph\'s primary use is to enable the creation of robust and complex AI agents by offering fine-grained control over application logic, unlike higher-level abstractions that might hide implementation details. \n\nKey capabilities include:\n*   **Durable Execution**: Agents can persist through failures and resume from their exact previous state. This is managed by `BaseCheckpointSaver` which stores `Checkpoin

## Trust and security notes for third-party MCP servers

Connecting to a server you don't control is not the same risk profile as
running your own stdio server, even when (like DeepWiki) it's free and
well-intentioned:

- **Treat every tool result as untrusted input.** A malicious or compromised
  MCP server could return text crafted to look like new instructions to the
  model ("prompt injection via tool output") -- the same discipline this
  repo's `security-check` skill applies to RAG/web-tool content applies
  here too.
- **A server's `list_tools()` descriptions are also untrusted.** Nothing
  stops a server from describing a tool misleadingly to bias which one gets
  picked -- don't wire an unfamiliar public server's tools into a
  high-stakes agent without reviewing what they actually do first.
- **Data volume is a real, demonstrated risk, not a hypothetical one** --
  this notebook just hit it. A server you don't control can return far more
  data than your context window (or your budget) can absorb; scope down to
  the specific tools you actually need, as done above.
- **Never send secrets to a public server.** No API keys, no internal data,
  no user PII -- treat the connection like any other third-party network
  call, because that's exactly what it is.
- **Prefer stdio (or an authenticated, access-controlled HTTP server you
  operate) for anything touching real business data.** Public HTTP MCP
  servers are best suited to genuinely public data, like this one.

## Explain like I'm 12

Notebooks 01-03 were like calling your own family members on a private
phone line only your house can use -- you trust whoever picks up because you
know exactly who's on the other end. This notebook is like calling a public
information hotline you found online -- it might genuinely be helpful and
free, but you wouldn't read out your house address or your parents' bank
details to whoever answers, and you'd double check that what they tell you
actually makes sense before repeating it as fact. Same phone, same protocol
for how the call works -- very different level of trust in who's on the
other end.

## Checkpoint questions

1. **Q: What's the one-line code difference between connecting to a local
   stdio server and this public HTTP server, using `MultiServerMCPClient`?**
   A: `{"command": ..., "args": ..., "transport": "stdio"}` becomes
   `{"url": "https://mcp.deepwiki.com/mcp", "transport": "streamable_http"}`
   -- everything downstream (`get_tools()`, binding to the LLM, `ToolNode`)
   is identical.

2. **Q: What real failure did this notebook hit, and what was the actual
   fix?**
   A: Binding all three DeepWiki tools let the model call
   `read_wiki_contents`, which returned ~167K tokens and exceeded
   gpt-4o-mini's context window. The fix was scoping the agent down to only
   `ask_question`, which returns a synthesized, appropriately-sized answer
   instead of a raw content dump.

3. **Q: Why is "the server might return too much data" specifically a
   third-party-server risk, and not really a concern with the calculator/
   knowledge_ops/orders servers from notebooks 01-03?**
   A: It's not exclusive to third-party servers technically, but you fully
   control your own servers' response sizes since you wrote them -- a
   public server's authors made their own choices about response size that
   you have no say in and must defend against on the client side instead.